# Ordinal and Multiclass Fire Risk Prediction on Aerial Imagery

In [ ]:
# Author: Leon Ohl
# Date: 10.4.2026

# This script implements and compares multiple deep learning and classical machine learning approaches for fire risk classification on the FireRisk dataset
# Dataset source: Hugging Face "blanchon/FireRisk"
# The script was excuted on Google Colab A100 GPU

# Content:
# 1. Imports
# 2. Understanding the dataset
# 3. Pytorch Custom Dataset
# 4. CNN Multiclass Classification with ResNet18
# 5. ViT Multiclass Classification with Swin_V2_B
# 6. Classical ML: SVC Transfer Learning
# 7. Ordinal Regression: Corn with ResNet18
# 8. Tensorboard

# Usage Notes:
# Run chapters 1 (Imports) and 3 (Dataset) before all other chapters
# Individual sections depend on previous components (see comments in beginning of chapters)
# If not running individual chapters, the script can be executed as a whole

# Limitations:
# Due to limited time towards the end of the project the code is not fully modularized (repeated components such as datamodules) and chapter dependencies are not wrapped into functions

## 1. Imports

In [ ]:
!pip install lightning
!pip install coral-pytorch

In [ ]:
%load_ext tensorboard

import torch
from lightning.pytorch import LightningDataModule, LightningModule, Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchmetrics import MetricCollection, MeanAbsoluteError
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassConfusionMatrix
)
from torchvision import transforms
from torchvision.transforms import v2
from torchvision.models import resnet18, swin_v2_b
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from einops import rearrange
from datasets import load_dataset
from collections import Counter
from lightning.pytorch.loggers import TensorBoardLogger
import time


# SVC
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.models.feature_extraction import get_graph_node_names
import numpy as np
from sklearn.model_selection import (
    GridSearchCV,
    PredefinedSplit
)
from sklearn.svm import LinearSVC
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.tensorboard import SummaryWriter

# CORN
from coral_pytorch.losses import corn_loss
from coral_pytorch.dataset import corn_label_from_logits


## 2. Understand the Dataset (don´t need to run this)

In [ ]:
test = load_dataset("blanchon/FireRisk") # import hugging face dataset
test["train"][0] # only train split exists in hf

In [ ]:
test["train"].features

In [ ]:
# check class imbalance
labels = test["train"]["label"]
counts = Counter(labels)

print(counts)

In [ ]:
# -> class imbalance order: very_low > non-burnable > low > moderate > high > very_high > water

## 3. Pytorch Custom Dataset

In [ ]:
# need to convert to torch dataset to return as torch tensor and to change label order

# order from hf: ['high'0, 'low'1, 'moderate'2, 'non-burnable'3, 'very_high'4, 'very_low'5, 'water'6])
# new order (for ordinal modeling): water 0, non-burnable 1, very_low 2, low 3, moderate 4, high 5, very_high 6
label_mapping = {
    6: 0,
    3: 1,
    5: 2,
    1: 3,
    2: 4,
    0: 5,
    4: 6
}

class FireRiskDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]['image']
        label_old = self.dataset[idx]['label']

        label = label_mapping[label_old] # sort labels in correct order for ordinal regression

        if self.transform:
            image = self.transform(image)

        return image, label

## 4. CNN Multiclass Classification with ResNet18

### 4.1 Datamodule

In [ ]:
class FireRiskDataModule(LightningDataModule):
  def __init__(self, batch_size: int = 128):
    super().__init__()
    self.batch_size = batch_size

    # resize to expected input size of resnet
    self.preprocess = v2.Compose([
        v2.Resize((224, 224)),
        v2.ToTensor(),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


    self.augmentation = v2.Compose([
      v2.RandomHorizontalFlip(),
      v2.RandomVerticalFlip(),
    ])

  def prepare_data(self):
    load_dataset("blanchon/FireRisk")

  def setup(self, stage):
    FireRisk_hf = load_dataset("blanchon/FireRisk")["train"] # only train split available in hf -> train split treated as full ds

    # only apply augmentation on train, changed after mistake in hw09
    # -> split indices
    dataset_size = len(FireRisk_hf)
    train_index, val_index, test_index = random_split(range(dataset_size), [0.6, 0.2, 0.2], generator=torch.Generator().manual_seed(42))

    train_dataset = FireRiskDataset(FireRisk_hf, transform=v2.Compose([self.augmentation, self.preprocess]))
    val_dataset = FireRiskDataset(FireRisk_hf, transform=self.preprocess)
    test_dataset = FireRiskDataset(FireRisk_hf, transform=self.preprocess)

    self.train_data = torch.utils.data.Subset(train_dataset, train_index.indices)
    self.val_data   = torch.utils.data.Subset(val_dataset, val_index.indices)
    self.test_data  = torch.utils.data.Subset(test_dataset, test_index.indices)


    # new ordered class names for plotting and conf matrix
    self.class_names = [
        "water",
        "non-burnable",
        "very_low",
        "low",
        "moderate",
        "high",
        "very_high"
    ]


    # to handle class imbalance: Inverse Class Frequency Method
    # inspiration: https://medium.com/@ravi.abhinav4/improving-class-imbalance-with-class-weights-in-machine-learning-af072fdd4aa4

    # consider new label order
    labels = []

    for old_label in FireRisk_hf['label']:
        new_label = label_mapping[old_label]
        labels.append(new_label)

    counts = Counter(labels)
    num_classes = 7
    total_samples = len(labels)

    # calculate class weights
    weights = []

    for i in range(num_classes):
        weight = total_samples / (num_classes * counts[i])
        weights.append(weight)

    self.class_weights = torch.tensor(weights) # convert to tensor for ce loss




  def train_dataloader(self):
    train_dl = DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True, num_workers=4)
    return train_dl

  # no shuffle
  def val_dataloader(self):
    val_dl = DataLoader(self.val_data, batch_size=self.batch_size, shuffle=False, num_workers=4)
    return val_dl

  def test_dataloader(self):
    test_dl = DataLoader(self.test_data, batch_size=self.batch_size, shuffle=False, num_workers=4)
    return test_dl

### 4.2 Model

In [ ]:
class LightningResnet18(LightningModule):
    def __init__(self, num_classes=7, class_weights=None, class_names=None):
        super().__init__()
        self.model = resnet18(weights='IMAGENET1K_V1')

        # replace final fc layer
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

        # ce loss function
        self.loss_fn = nn.CrossEntropyLoss(weight=class_weights) # pass class_weights from datamodule to handle imbalance

        # MAE for comparison with ordinal regression
        self.valid_mae = MeanAbsoluteError()
        self.test_mae = MeanAbsoluteError()

        # classification metrics
        self.train_metrics = MetricCollection(
            {
                "overall_accuracy": MulticlassAccuracy(num_classes=num_classes, average="micro"),
                "accuracy": MulticlassAccuracy(num_classes=num_classes),
                "f1": MulticlassF1Score(num_classes=num_classes),
                "precision": MulticlassPrecision(num_classes=num_classes),
                "recall": MulticlassRecall(num_classes=num_classes),
            },
            prefix="train_",
        )
        self.valid_metrics = self.train_metrics.clone(prefix="valid_")
        self.test_metrics = self.train_metrics.clone(prefix="test_")

        self.val_confusion = MulticlassConfusionMatrix(num_classes=num_classes, normalize="true") # normalized confusion matrix; see pytorch lightning doc

        self.class_names = class_names


    # forward pass
    def forward(self, inputs):
        return self.model(inputs)


    def training_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.train_metrics.update(output, target)

        # logged loss to understand problem of rapid validation_accuracy decrease
        loss = self.loss_fn(output, target)
        self.log("train_loss", loss, on_step=True, on_epoch=False, prog_bar=True)

        return loss

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), on_step=False, on_epoch=True)
        self.train_metrics.reset()

    def validation_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.valid_metrics.update(output, target)

        loss = self.loss_fn(output, target)
        self.log("valid_loss", loss, on_step=True, on_epoch=False, prog_bar=True)


        # confusion matrix
        # inspiration: https://stackoverflow.com/questions/65498782/how-to-dump-confusion-matrix-using-tensorboard-logger-in-pytorch-lightning
        predicted_labels = torch.argmax(output, dim=1) # get predicted labels (index)

        self.val_confusion.update(predicted_labels, target)


        # MAE
        self.valid_mae.update(predicted_labels, target)


    def on_validation_epoch_end(self):
        self.log_dict(self.valid_metrics.compute(), on_step=False, on_epoch=True)
        self.valid_metrics.reset()

        self.log("valid_mae", self.valid_mae.compute(), on_epoch=True, on_step=False)
        self.valid_mae.reset()


        # Confusion matrix plotting
        tensorboard_logger = self.logger.experiment

        fig, ax = plt.subplots(figsize=(8,8))

        self.val_confusion.plot(ax=ax, labels=self.class_names)

        plt.xlabel("Predicted Class")
        plt.ylabel("True Class")
        plt.title("Validation Confusion Matrix")
        plt.tight_layout()

        # log to tensorboard
        tensorboard_logger.add_figure("validation_confusion_matrix", fig, self.current_epoch)
        plt.close(fig)
        self.val_confusion.reset()

    def test_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.test_metrics.update(output, target)

        # MAE
        predicted_labels = torch.argmax(output, dim=1)
        self.test_mae.update(predicted_labels, target)

    def on_test_epoch_end(self):
        self.log_dict(self.test_metrics.compute(), on_step=False, on_epoch=True)
        self.test_metrics.reset()

        # MAE
        self.log("test_mae", self.test_mae.compute(), on_step=False, on_epoch=True)
        self.test_mae.reset()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.model.parameters(), lr=1e-4)  # lr decreased after overshooting problem
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": ReduceLROnPlateau(optimizer, mode='max'),
                "monitor": "valid_accuracy"
            },
        }

### 4.3 Callbacks

In [ ]:
def get_callbacks(run_name):
    early_stopping = EarlyStopping(
        monitor='valid_accuracy',
        mode='max'
    )
    # monitor valid accuracy, good for imbalanced datasets
    # standard patience sufficient for model, bc val_loss increasing trend, so no better predictions expected if running longer

    checkpoint_callback = ModelCheckpoint(
        dirpath=f'/content/checkpoints/{run_name}',
        monitor='valid_accuracy',
        mode='max',
        save_top_k=1
    )

    return [early_stopping, checkpoint_callback]

### 4.4 Training and Testing

In [ ]:
# implemented separate runs for unweighted and weighted model to compare in tensorboard

In [ ]:
datamodule = FireRiskDataModule()
datamodule.setup("fit") # call setup before passing class_weights to model
# https://lightning.ai/docs/pytorch/stable/data/datamodule.html#using-a-datamodule

4.4.1 Unweighted Model

In [ ]:
run = "unweighted_resnet"

callbacks_unweighted = get_callbacks(run)
checkpoint_unweighted = callbacks_unweighted[1]

logger_unweighted = TensorBoardLogger(save_dir="/content", name="models", version=run)

model_unweighted = LightningResnet18(class_weights=None, class_names=datamodule.class_names)

In [ ]:
fast_dev_run=False # for testing
trainer_unweighted = Trainer(fast_dev_run=fast_dev_run, max_epochs=10, logger=logger_unweighted, callbacks=callbacks_unweighted)

# show runtime; inspiration:  https://raschka-research-group.github.io/coral-pytorch/tutorials/pytorch_lightning/ordinal-corn_mnist/
start_time = time.time()
trainer_unweighted.fit(model_unweighted, datamodule)

runtime = (time.time() - start_time)/60
print(f"Training took {runtime:.2f} min in total.")

In [ ]:
# checkpointing to get best model because of overfitting in later epochs
best_unweighted = checkpoint_unweighted.best_model_path
trainer_unweighted.test(datamodule=datamodule, ckpt_path=best_unweighted)

Weigthed Model

In [ ]:
run = "weighted_resnet"

callbacks_weighted = get_callbacks(run)
checkpoint_weighted = callbacks_weighted[1]

logger_weighted = TensorBoardLogger(save_dir="/content", name="models", version=run)

model_weighted = LightningResnet18(class_weights=datamodule.class_weights, class_names=datamodule.class_names)

In [ ]:
fast_dev_run=False
trainer_weighted = Trainer(fast_dev_run=fast_dev_run, max_epochs=10, logger=logger_weighted, callbacks=callbacks_weighted)

start_time = time.time()
trainer_weighted.fit(model_weighted, datamodule)

runtime = (time.time() - start_time)/60
print(f"Training took {runtime:.2f} min in total.")

In [ ]:
best_weighted = checkpoint_weighted.best_model_path
trainer_weighted.test(datamodule=datamodule, ckpt_path=best_weighted)

### 4.5 Notes

In [ ]:
# removed example image plotting (prediction and true label) from code to speed up (only needed for midterm report)
# class imbalance led to overfitting on non-burnable class, which is second most dominant but easy-to-predict class
# after some epochs valid acc was decreasing rapidly, while train loss increased. early stopping was triggered
# countermeasures:
# lr was reduced to 1e-4 to avoid overshooting
# augmentation added
# class weights

## 5. ViT Multiclass Classification with Swin_V2_B

In [ ]:
# mostly copied code from chapter 1; changed lines are marked with comments
# run chapter 1, 3, 4.1, 4.3 before running chapter 5

### 5.1 Datamodule

In [ ]:
# run chapter 4.1 datamodule

### 5.2 Model

In [ ]:
class LightningSwin(LightningModule):
    def __init__(self, num_classes=7, class_weights=None, class_names=None):
        super().__init__()
        self.model = swin_v2_b(weights='IMAGENET1K_V1')

        # replace classification head instead of final fc layer like in resnet architecture
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)

        self.loss_fn = nn.CrossEntropyLoss(weight=class_weights)

        self.valid_mae = MeanAbsoluteError()
        self.test_mae = MeanAbsoluteError()

        self.train_metrics = MetricCollection(
            {
                "overall_accuracy": MulticlassAccuracy(num_classes=num_classes, average="micro"),
                "accuracy": MulticlassAccuracy(num_classes=num_classes),
                "f1": MulticlassF1Score(num_classes=num_classes),
                "precision": MulticlassPrecision(num_classes=num_classes),
                "recall": MulticlassRecall(num_classes=num_classes),
            },
            prefix="train_",
        )
        self.valid_metrics = self.train_metrics.clone(prefix="valid_")
        self.test_metrics = self.train_metrics.clone(prefix="test_")

        self.val_confusion = MulticlassConfusionMatrix(num_classes=num_classes, normalize="true")

        self.class_names = class_names


    def forward(self, inputs):
        return self.model(inputs)


    def training_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.train_metrics.update(output, target)

        loss = self.loss_fn(output, target)
        self.log("train_loss", loss, on_step=True, on_epoch=False, prog_bar=True)

        return loss

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), on_step=False, on_epoch=True)
        self.train_metrics.reset()

    def validation_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.valid_metrics.update(output, target)

        loss = self.loss_fn(output, target)
        self.log("valid_loss", loss, on_step=True, on_epoch=False, prog_bar=True)



        predicted_labels = torch.argmax(output, dim=1)
        self.val_confusion.update(predicted_labels, target)


        self.valid_mae.update(predicted_labels, target)


    def on_validation_epoch_end(self):
        self.log_dict(self.valid_metrics.compute(), on_step=False, on_epoch=True)
        self.valid_metrics.reset()

        self.log("valid_mae", self.valid_mae.compute(), on_epoch=True, on_step=False)
        self.valid_mae.reset()

        tensorboard_logger = self.logger.experiment


        fig, ax = plt.subplots(figsize=(8,8))

        self.val_confusion.plot(ax=ax, labels=self.class_names)

        plt.xlabel("Predicted Class")
        plt.ylabel("True Class")
        plt.title("Validation Confusion Matrix")
        plt.tight_layout()

        tensorboard_logger.add_figure("validation_confusion_matrix", fig, global_step=self.current_epoch)
        plt.close(fig)
        self.val_confusion.reset()


    def test_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        self.test_metrics.update(output, target)

        predicted_labels = torch.argmax(output, dim=1)
        self.test_mae.update(predicted_labels, target)

    def on_test_epoch_end(self):
        self.log_dict(self.test_metrics.compute(), on_step=False, on_epoch=True)
        self.test_metrics.reset()

        self.log("test_mae", self.test_mae.compute(), on_step=False, on_epoch=True)
        self.test_mae.reset()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.model.parameters(), lr=1e-4)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": ReduceLROnPlateau(optimizer, mode='max'),
                "monitor": "valid_accuracy"
            },
        }

### 5.3 Callbacks

In [ ]:
# run chapter 4.3 callbacks

### 5.4 Training and Testing

In [ ]:
datamodule = FireRiskDataModule(batch_size=32) # reduced batch size bc swin bigger
datamodule.setup("fit")

In [ ]:
run = "weighted_swin"

callbacks_swin = get_callbacks(run)
checkpoint_swin = callbacks_swin[1]

logger_swin = TensorBoardLogger(save_dir="/content", name="models", version=run)

model_swin = LightningSwin(class_weights=datamodule.class_weights, class_names=datamodule.class_names)

In [ ]:
fast_dev_run=False
trainer_swin = Trainer(fast_dev_run=fast_dev_run, max_epochs=10, logger=logger_swin, callbacks=callbacks_swin)

start_time = time.time()
trainer_swin.fit(model_swin, datamodule)

runtime = (time.time() - start_time)/60
print(f"Training took {runtime:.2f} min in total.")

In [ ]:
best_swin = checkpoint_swin.best_model_path
trainer_swin.test(datamodule=datamodule, ckpt_path=best_swin)

## 6. Classical ML: SVC Transfer Learning

In [ ]:
# this chapter builds up on chapter 5
# therefore run chapter 5 before running chapter 6

### 6.1 DL Model

In [ ]:
datamodule = FireRiskDataModule()
datamodule.setup(stage="fit")

In [ ]:
# load swin_v2_b model trained on FireRisk in chapter 5
model = LightningSwin.load_from_checkpoint(best_swin, class_weights=datamodule.class_weights, class_names=datamodule.class_names)

# see hw06, only forward loop to extract features
model.to("cuda")
model.eval()

### 6.2 Feature Extractor

In [ ]:
# print node names to know last layer
# https://pytorch.org/blog/fx-feature-extraction-torchvision/

nodes, _ = get_graph_node_names(model.model)
print(nodes)

In [ ]:
return_nodes = {'flatten': 'features'}  # get last layer before fc layer
feature_extractor = create_feature_extractor(model.model, return_nodes=return_nodes)


# check feature vector dimensions
# https://docs.pytorch.org/vision/main/generated/torchvision.models.feature_extraction.create_feature_extractor.html#torchvision.models.feature_extraction.create_feature_extractor

out = feature_extractor(torch.rand(1, 3, 224, 224).to("cuda")) # input example image
print([(k, v.shape) for k, v in out.items()]) # doc -> svc expects X: {array-like, sparse matrix} of shape (n_samples, n_features)

### 6.3 Extract features FireRisk data (on GPU)

In [ ]:
# (https://lightning.ai/docs/pytorch/stable/data/datamodule.html#datamodules-without-lightning)

train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()
test_loader = datamodule.test_dataloader()

6.3.1 Train dl

In [ ]:
# run dataset through feature extractor
all_features = []
all_labels = []


# (https://yassin01.medium.com/understanding-the-difference-between-model-eval-and-model-train-in-pytorch-48e3002ee0a2)

with torch.no_grad(): # (https://discuss.pytorch.org/t/hybrid-cnn-svm-feature-extraction/198833)
    for images, labels in train_loader:
        images = images.to("cuda")
        out = feature_extractor(images)
        features = out['features']
        all_features.append(features)
        all_labels.append(labels)

In [ ]:
print(all_features[0].shape)

In [ ]:
X_train = torch.cat(all_features)
y_train = torch.cat(all_labels)

In [ ]:
X_train.shape

6.3.2 Val dl

In [ ]:
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to("cuda")
        out = feature_extractor(images)
        features = out['features']
        all_features.append(features)
        all_labels.append(labels)

X_val = torch.cat(all_features)
y_val = torch.cat(all_labels)

6.3.3 Test dl

In [ ]:
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to("cuda")
        out = feature_extractor(images)
        features = out['features']
        all_features.append(features)
        all_labels.append(labels)

X_test = torch.cat(all_features)
y_test = torch.cat(all_labels)

### 6.4 SVC (on CPU)

6.4.1 Hyperparameter Tuning

In [ ]:
# extracted features on cpu and torch tensor conversion to numpy
X_train = X_train.cpu().numpy()
y_train = y_train.cpu().numpy()
X_val = X_val.cpu().numpy()
y_val = y_val.cpu().numpy()
X_test = X_test.cpu().numpy()
y_test = y_test.cpu().numpy()

In [ ]:
X_train.shape

In [ ]:
# feature scaling recommended in SVM doc and https://www.baeldung.com/cs/svm-feature-scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
estimator = LinearSVC(class_weight='balanced') # balanced class weights bc class imbalance

# following code more or less copied from hw04

# gridsearch.fit only has arguments x,y -> fuse dataframes and tell in cv how to train/val split
X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

X_train_val.shape[0]

In [ ]:
# provides train and val indices by setting -1 for all train data and 0 for all validation data
test_fold = np.concatenate([
    -1 * np.ones(len(X_train)),
     0 * np.ones(len(X_val))
])

crossval = PredefinedSplit(test_fold)

In [ ]:
# chose only 1 param, otherwise took too long
parameters = {
    "C": [0.01, 0.1]
    }

In [ ]:
start_time = time.time()
gridsearch = GridSearchCV(estimator, parameters, cv=crossval)

gridsearch.fit(X_train_val, y_train_val)

runtime = (time.time() - start_time)/60
print(f"Gridsearch took {runtime:.2f} min in total.")

In [ ]:
gridsearch.best_params_

6.4.2 Evaluation

In [ ]:
# see hw03/04
y_pred = gridsearch.best_estimator_.predict(X_test)

In [ ]:
oa = accuracy_score(y_test, y_pred)
aa = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
p = precision_score(y_test, y_pred, average='macro')
r = recall_score(y_test, y_pred, average='macro')

print(f"Overall Accuracy: {oa}")
print(f"Average Accuracy: {aa}")
print(f"Precision: {p}")
print(f"Recall: {r}")
print(f"F1-score: {f1}")


In [ ]:
# log to tensorboard for comparison
writer = SummaryWriter("/content/models/svc")

writer.add_scalar("test_overall_accuracy", oa)
writer.add_scalar("test_accuracy", aa)
writer.add_scalar("test_f1", f1)
writer.add_scalar("test_precision", p)
writer.add_scalar("test_recall", r)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, normalize='true', display_labels=datamodule.class_names, values_format='.2f', xticks_rotation='vertical', ax=ax)

writer.add_figure("test_confusion_matrix", fig)
plt.close(fig)

## 7. Ordinal Regression: CORN with Resnet18

In [ ]:
# run chapter 1, 3, 4.1 before running chapter 7
# Tutorial: https://raschka-research-group.github.io/coral-pytorch/tutorials/pytorch_lightning/ordinal-corn_mnist/

### 7.1 Datamodule

In [ ]:
# run chapter 4.1 datamodule

### 7.2 Model

In [ ]:
# copied from chapter 4.2, changed lines are commented
class LightningResnet18(LightningModule):
    def __init__(self, num_classes=7, class_names=None): # no class weights bc no ce loss
        super().__init__()
        self.model = resnet18(weights='IMAGENET1K_V1')

        # reduce number of classes in output layer
        self.num_classes = num_classes
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes-1)


        # no ce loss bc corn loss function is used now

        # MAE to evaluate regression
        self.train_mae = MeanAbsoluteError()
        self.valid_mae = MeanAbsoluteError()
        self.test_mae = MeanAbsoluteError()


        self.train_metrics = MetricCollection(
            {
                "overall_accuracy": MulticlassAccuracy(num_classes=num_classes, average="micro"),
                "accuracy": MulticlassAccuracy(num_classes=num_classes),
                "f1": MulticlassF1Score(num_classes=num_classes),
                "precision": MulticlassPrecision(num_classes=num_classes),
                "recall": MulticlassRecall(num_classes=num_classes),
            },
            prefix="train_",
        )
        self.valid_metrics = self.train_metrics.clone(prefix="valid_")
        self.test_metrics = self.train_metrics.clone(prefix="test_")

        self.val_confusion = MulticlassConfusionMatrix(num_classes=num_classes, normalize="true")
        self.class_names = class_names


    def forward(self, inputs):
        return self.model(inputs)


    def training_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        # CORN logits to labels
        predicted_labels = corn_label_from_logits(output)

        self.train_metrics.update(predicted_labels, target) # metrics on predicted_labels instead logits

        # MAE
        self.train_mae.update(predicted_labels, target)


        # corn loss, pass num_classes
        loss = corn_loss(output, target, num_classes=self.num_classes)
        self.log("train_loss", loss, on_step=True, on_epoch=False, prog_bar=True)

        return loss

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), on_step=False, on_epoch=True)
        self.train_metrics.reset()

        # MAE
        self.log("train_mae", self.train_mae.compute(), on_epoch=True, on_step=False)
        self.train_mae.reset()

    def validation_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        # CORN logits to labels
        predicted_labels = corn_label_from_logits(output)

        # use new predicted_labels for confmatrix
        self.val_confusion.update(predicted_labels, target)

        # metrics on predicted_labels instead logits
        self.valid_metrics.update(predicted_labels, target)

        # MAE
        self.valid_mae.update(predicted_labels, target)

        # pass num_classes
        loss = corn_loss(output, target, num_classes=self.num_classes)
        self.log("valid_loss", loss, on_step=True, on_epoch=False, prog_bar=True)


    def on_validation_epoch_end(self):
        self.log_dict(self.valid_metrics.compute(), on_step=False, on_epoch=True)
        self.valid_metrics.reset()

        # MAE
        self.log("valid_mae", self.valid_mae.compute(), on_epoch=True, on_step=False)
        self.valid_mae.reset()



        tensorboard_logger = self.logger.experiment

        fig, ax = plt.subplots(figsize=(8,8))

        self.val_confusion.plot(ax=ax, labels=self.class_names)

        plt.xlabel("Predicted Class")
        plt.ylabel("True Class")
        plt.title("Validation Confusion Matrix")
        plt.tight_layout()

        tensorboard_logger.add_figure("validation_confusion_matrix", fig, global_step=self.current_epoch)

        plt.close(fig)
        self.val_confusion.reset()


    def test_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.model(inputs)

        # CORN logits to labels
        predicted_labels = corn_label_from_logits(output)

        # metrics on predicted_labels instead logits
        self.test_metrics.update(predicted_labels, target)

        # MAE
        self.test_mae.update(predicted_labels, target)


    def on_test_epoch_end(self):
        self.log_dict(self.test_metrics.compute(), on_step=False, on_epoch=True)
        self.test_metrics.reset()

        # MAE
        self.log("test_mae", self.test_mae.compute())
        self.test_mae.reset()


    # monitor validation MAE
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.model.parameters(), lr=1e-4)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": ReduceLROnPlateau(optimizer, mode='min'),
                "monitor": "valid_mae"
            },
        }

### 7.3 Callbacks

In [ ]:
# monitor validation MAE (min)
def get_callbacks(run_name):
    early_stopping = EarlyStopping(
        monitor='valid_mae',
        mode='min'
    )


    checkpoint_callback = ModelCheckpoint(
        dirpath=f'/content/checkpoints/{run_name}',
        monitor='valid_mae',
        mode='min',
        save_top_k=1
    )

    return [early_stopping, checkpoint_callback]

### 7.4 Training and Testing

In [ ]:
datamodule = FireRiskDataModule()
datamodule.setup("fit")

run = "corn"

callbacks_corn = get_callbacks(run)
checkpoint_corn = callbacks_corn[1]

logger_corn = TensorBoardLogger(save_dir="/content", name="models", version=run)

model_corn = LightningResnet18(class_names=datamodule.class_names) # no class weights

In [ ]:
fast_dev_run=False

trainer_corn = Trainer(fast_dev_run=fast_dev_run, max_epochs=10, logger=logger_corn, callbacks=callbacks_corn)

start_time = time.time()
trainer_corn.fit(model_corn, datamodule)

runtime = (time.time() - start_time)/60
print(f"Training took {runtime:.2f} min in total.")

In [ ]:
best_corn = checkpoint_corn.best_model_path
trainer_corn.test(datamodule=datamodule, ckpt_path=best_corn)

7.4.1 Performance baseline

In [ ]:
all_test_labels = []

test_loader = datamodule.test_dataloader()

for images, labels in test_loader:
    all_test_labels.append(labels)
all_test_labels = torch.cat(all_test_labels)

all_test_labels = all_test_labels.float()
avg_prediction = torch.median(all_test_labels)
baseline_mae = torch.mean(torch.abs(all_test_labels - avg_prediction))
print(f'Baseline MAE: {baseline_mae:.2f}') # from doc:a model that would always predict the dataset median would achieve the Baseline MAE

## 8. Tensorboard

In [ ]:
%tensorboard --logdir /content/models